# AI-Powered Lead Routing Agent

## Overview
This agent automatically routes incoming B2B leads to the correct sales rep 
based on territory, company size, lead score, and existing account ownership rules.

## Routing Logic
- **Territory Rules** — routes by country/region to the assigned rep
- **Company Size Override** — 1000+ employees always go to Enterprise team
- **Lead Score Override** — score 80+ triggers HOT priority flag
- **Existing Account Rule** — if account owner exists, route to them regardless of territory

## Output Format
- **Assigned To** — rep name and email
- **Queue** — Enterprise / Mid-Market / SMB
- **Priority** — Hot / Normal
- **Reason** — full explanation of routing decision
- **Next SLA** — same day / 24 hours / 48 hours

## Real-World Application
Mirrors LeanData routing logic — territory assignments, 
account ownership rules, and priority escalation based on lead score.

## Tech Stack
- Python
- Anthropic Claude API (claude-haiku)
- Jupyter Notebook

## Author
Karthik Ayyer — Marketing Automation Manager| Marketing Ops Manager | MarTech & Revenue Operations

In [1]:
!pip install anthropic python-dotenv -q

In [2]:
from dotenv import load_dotenv
import os
import anthropic

load_dotenv()
client = anthropic.Anthropic()

In [3]:
routing_rules = """
You are a B2B SaaS lead routing expert. Route the incoming lead to the correct sales rep based on these rules:

TERRITORY RULES:
- India & APAC → Ravi Kumar (ravi@company.com)
- USA & Canada → Sarah Chen (sarah@company.com)
- Europe (UK, DE, FR, etc.) → James O'Brien (james@company.com)
- Rest of World → Sarah Chen (sarah@company.com)

COMPANY SIZE OVERRIDE:
- 1000+ employees anywhere → Enterprise team → Michael Scott (michael@company.com)

LEAD SCORE OVERRIDE:
- Score 80+ → Senior rep in their territory (add PRIORITY: HOT flag)

EXISTING ACCOUNT RULE:
- If account_owner field is set → route to that owner regardless of territory

Return your response in this format:
ASSIGNED TO: [rep name and email]
QUEUE: [Enterprise / Mid-Market / SMB]
PRIORITY: [Hot / Normal]
REASON: [explain the routing decision]
NEXT SLA: [when the rep should follow up — same day / 24 hours / 48 hours]
"""

In [4]:
leads = [
    {
        "name": "Rahul Sharma",
        "company": "TechCorp India",
        "country": "India",
        "employees": 500,
        "lead_score": 85,
        "industry": "SaaS",
        "account_owner": ""
    },
    {
        "name": "Sarah Johnson",
        "company": "Enterprise Corp",
        "country": "USA",
        "employees": 2000,
        "lead_score": 72,
        "industry": "Financial Services",
        "account_owner": ""
    },
    {
        "name": "Hans Mueller",
        "company": "Berlin Startup",
        "country": "Germany",
        "employees": 150,
        "lead_score": 60,
        "industry": "FinTech",
        "account_owner": ""
    },
    {
        "name": "Priya Patel",
        "company": "MidMarket Inc",
        "country": "India",
        "employees": 800,
        "lead_score": 45,
        "industry": "Healthcare",
        "account_owner": "ravi@company.com"
    },
    {
        "name": "James Wilson",
        "company": "SmallBiz Co",
        "country": "USA",
        "employees": 30,
        "lead_score": 55,
        "industry": "Retail",
        "account_owner": ""
    }
]

In [11]:
for lead in leads:
    print(f"\n{'='*50}")
    print(f"Routing Lead: {lead['name']} - {lead['company']} {lead['country']}")
    print(f"{'='*50}")

    message = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[
            {
                "role":"user",
                "content": f"{routing_rules}\n\nRoute this lead:\n{lead}"
            }
        ]
    )
    print(message.content[0].text)
            


Routing Lead: Rahul Sharma - TechCorp India India
ASSIGNED TO: Ravi Kumar (ravi@company.com)
QUEUE: Mid-Market
PRIORITY: Hot
REASON: Lead is located in India (territory match for Ravi Kumar). Lead score of 85 triggers the LEAD SCORE OVERRIDE rule, elevating to senior rep in territory with HOT priority flag. Company size (500 employees) does not trigger Enterprise override (requires 1000+). No existing account owner assigned.
NEXT SLA: Same day

Routing Lead: Sarah Johnson - Enterprise Corp USA
ASSIGNED TO: Michael Scott (michael@company.com)
QUEUE: Enterprise
PRIORITY: Normal
REASON: Company size override applies - Enterprise Corp has 2000 employees, which triggers automatic routing to the Enterprise team regardless of territory (USA) or lead score (72).
NEXT SLA: Same day

Routing Lead: Hans Mueller - Berlin Startup Germany
ASSIGNED TO: James O'Brien (james@company.com)
QUEUE: Mid-Market
PRIORITY: Normal
REASON: Lead is located in Germany (Europe territory), which routes to James O'B